# 1. Preprocessing Data Kuesioner Mahasiswa (KESMA PENS)
Tahap ini bertugas membaca data mentah Google Form, membersihkan demografi, dan mengonversi skala Likert ke skala 0 - 100.

In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

print('Library berhasil di-import!')

Library berhasil di-import!


In [3]:
# 1. Cari dan load file survei mentah
file_path = None
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    matches = list(p.glob('**/*SURVEI KESMA*.csv'))
    if matches:
        file_path = matches[0]
        break

if not file_path:
    raise FileNotFoundError('File survei KESMA tidak ditemukan!')

print(f'Memuat data dari: {file_path}')
df_raw = pd.read_csv(file_path)
print(f'Dimensi data mentah: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom')

Memuat data dari: c:\Tugas Pens Ardan\semester 5\projek analitika data\nadi kampus web\SURVEI KESMA (Responses) - Form responses 1.csv
Dimensi data mentah: 114 baris x 44 kolom


In [4]:
# Pratinjau 3 data teratas
df_raw.head(3)

,Timestamp,Jenis Kelamin,Semester saat ini,Angkatan,Departemen,Program Studi,"Dalam beberapa minggu terakhir, saya merasa mampu menjalani aktivitas perkuliahan dan kegiatan sehari-hari dengan baik.","Saya mampu mengelola tekanan yang muncul dari tugas, praktikum, ujian, atau kegiatan akademik lainnya.",Saya memiliki waktu yang cukup untuk beristirahat dan memulihkan energi di tengah aktivitas perkuliahan.,Saya masih memiliki semangat dan motivasi untuk mengikuti kegiatan perkuliahan.,...,Saya memiliki gambaran yang cukup jelas mengenai pekerjaan atau karier yang ingin saya jalani setelah lulus.,Saya merasa kompetensi yang saya miliki cukup relevan dengan kebutuhan dunia kerja.,"Saya memiliki kesempatan memperoleh pengalaman melalui proyek, magang, kompetisi, organisasi, atau kegiatan lain yang mendukung kesiapan kerja.","Saya mudah memperoleh informasi mengenai magang, lowongan pekerjaan, pengembangan kompetensi, atau peluang karier.",Saya memiliki pemahaman yang cukup mengenai kompetensi dan kemampuan yang dibutuhkan di dunia kerja.,"Saya cukup percaya diri untuk menghadapi proses magang, rekrutmen, atau memasuki dunia kerja.","Saya mengetahui layanan, kegiatan, atau pihak di kampus yang dapat membantu persiapan karier saya.","Secara keseluruhan, saya merasa cukup siap menghadapi dunia kerja setelah menyelesaikan studi.",Dukungan apa yang paling Anda butuhkan dari kampus untuk meningkatkan kesiapan menghadapi dunia kerja?,Saya memiliki kondisi ekonomi yang mendukung
0,22/09/2026 14:32:52,Laki-laki,Semester 5,2024,Departemen Teknik Elektro (DTE),S. Tr. Teknik Elektro Industri,4,4,3,3,...,3,3,3,3,3,4,3,3,Pelatihan yang akan dibuat bekal di dunia kerja,NaN
1,22/09/2026 14:35:08,Perempuan,Semester 3,2025,Departemen Teknologi Multimedia Kreatif (DTMK),D3 Teknologi Multimedia dan Broadcasting,1,4,1,2,...,2,1,2,1,1,1,1,1,mungkin dukungan mental? bingung jugaa,NaN
2,22/09/2026 14:35:27,Laki-laki,Semester 5,2024,Departemen Teknik Informatika dan Komputer (DTIK),S.Tr. Teknik Informatika,3,4,4,4,...,1,3,4,2,3,4,3,2,Koneksi orang dalam,NaN


In [5]:
# Pengecekan missing values dan duplikasi
print(f'Jumlah data duplikat: {df_raw.duplicated().sum()}')
print('\nKolom dengan missing values:')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

Jumlah data duplikat: 0

Kolom dengan missing values:
Dukungan apa yang paling Anda butuhkan dari kampus untuk meningkatkan kesiapan menghadapi dunia kerja?      1
Saya memiliki kondisi ekonomi yang mendukung                                                              114
dtype: int64


In [6]:
# 2. Preprocessing Data Demografi & Standardisasi Program Studi
def standardize_prodi(val):
    if pd.isna(val): return 'Lainnya'
    s = str(val).strip().lower()
    if 'sains data' in s or 'data' in s: return 'Sains Data Terapan'
    elif 'informatika' in s: return 'Teknik Informatika'
    elif 'elektro' in s: return 'Teknik Elektro Industri'
    elif 'telekomunikasi' in s or 'telkom' in s: return 'Teknik Telekomunikasi'
    elif 'mekatronika' in s: return 'Teknik Mekatronika'
    elif 'multimedia' in s or 'broadcasting' in s or 'mmb' in s: return 'Teknologi Multimedia Broadcasting'
    elif 'game' in s: return 'Teknologi Game'
    elif 'sistem informasi' in s or 'is' in s: return 'Sistem Informasi'
    elif 'komputer' in s: return 'Teknik Komputer'
    else: return str(val).strip()

df_clean = pd.DataFrame()
df_clean['student_id'] = [f'MHS-{i+1:03d}' for i in range(len(df_raw))]
df_clean['gender'] = df_raw['Jenis Kelamin'].fillna('Lainnya').str.strip()
df_clean['semester'] = df_raw['Semester saat ini'].astype(str).str.extract(r'(\d+)').fillna(5).astype(int)
df_clean['prodi'] = df_raw['Program Studi'].apply(standardize_prodi)
df_clean.head()

,student_id,gender,semester,prodi
0,MHS-001,Laki-laki,5,Teknik Elektro Industri
1,MHS-002,Perempuan,3,Teknologi Multimedia Broadcasting
2,MHS-003,Laki-laki,5,Teknik Informatika
3,MHS-004,Laki-laki,5,Sains Data Terapan
4,MHS-005,Laki-laki,5,Teknik Informatika


In [7]:
# 3. Fungsi konversi skala Likert 1-5 ke skala standar 0-100
def to_score(series):
    num = pd.to_numeric(series, errors='coerce').fillna(3)
    return ((num - 1) / 4.0) * 100.0

In [8]:
# Skoring Dimensi 1: Wellbeing Index (Pertanyaan kolom 6 s.d. 14)
wb_cols = df_raw.columns[6:15]
df_clean['wellbeing'] = df_raw[wb_cols].apply(to_score).mean(axis=1).round(1)
print('Statistik Dimensi Wellbeing Index:')
print(df_clean['wellbeing'].describe().round(2))

Statistik Dimensi Wellbeing Index:
count    114.00
mean      60.31
std       19.11
min        8.30
25%       47.20
50%       58.30
75%       75.00
max      100.00
Name: wellbeing, dtype: float64


In [9]:
# Skoring Dimensi 2: Academic Pressure (Pertanyaan kolom 16 s.d. 23)
# Dihitung: 100 - daya kelola beban tugas agar skor tinggi = tekanan akademik tinggi
acad_cols = df_raw.columns[16:24]
acad_coping = df_raw[acad_cols].apply(to_score).mean(axis=1)
df_clean['academic_pressure'] = (100.0 - acad_coping).round(1)
print('Statistik Dimensi Academic Pressure:')
print(df_clean['academic_pressure'].describe().round(2))

Statistik Dimensi Academic Pressure:
count    114.00
mean      40.13
std       18.04
min        0.00
25%       28.10
50%       40.60
75%       50.00
max       78.10
Name: academic_pressure, dtype: float64


In [10]:
# Skoring Dimensi 3: Social Support (Pertanyaan kolom 25 s.d. 32)
soc_cols = df_raw.columns[25:33]
df_clean['social_support'] = df_raw[soc_cols].apply(to_score).mean(axis=1).round(1)
print('Statistik Dimensi Social Support:')
print(df_clean['social_support'].describe().round(2))

Statistik Dimensi Social Support:
count    114.00
mean      73.30
std       18.41
min       18.80
25%       62.50
50%       75.00
75%       84.40
max      100.00
Name: social_support, dtype: float64


In [11]:
# Skoring Dimensi 4: Career Readiness (Pertanyaan kolom 34 s.d. 41)
car_cols = df_raw.columns[34:42]
df_clean['career_readiness'] = df_raw[car_cols].apply(to_score).mean(axis=1).round(1)
print('Statistik Dimensi Career Readiness:')
print(df_clean['career_readiness'].describe().round(2))

Statistik Dimensi Career Readiness:
count    114.00
mean      56.47
std       21.12
min        3.10
25%       44.58
50%       53.10
75%       71.90
max      100.00
Name: career_readiness, dtype: float64


In [12]:
# Ekstraksi teks aspirasi/keluhan terbuka
df_clean['keluhan_akademik'] = df_raw.iloc[:, 24].fillna('').astype(str).str.strip()
df_clean['kebutuhan_karier'] = df_raw.iloc[:, 42].fillna('').astype(str).str.strip()

print('Dataframe hasil preprocessing lengkap:')
df_clean.head()

Dataframe hasil preprocessing lengkap:


,student_id,gender,semester,prodi,wellbeing,academic_pressure,social_support,career_readiness,keluhan_akademik,kebutuhan_karier
0,MHS-001,Laki-laki,5,Teknik Elektro Industri,66.7,43.8,75.0,53.1,Fasilitas laboratorium yang mungkin harus lebi...,Pelatihan yang akan dibuat bekal di dunia kerja
1,MHS-002,Perempuan,3,Teknologi Multimedia Broadcasting,41.7,53.1,59.4,6.2,"merasa kapasitas otak tidak mumpuni, yang akhi...",mungkin dukungan mental? bingung jugaa
2,MHS-003,Laki-laki,5,Teknik Informatika,61.1,40.6,59.4,43.8,Wifi PENS busuk,Koneksi orang dalam
3,MHS-004,Laki-laki,5,Sains Data Terapan,63.9,31.2,68.8,59.4,Dosen kurang disiplin dan visi,Informasi skills atau pekerjaan yang kebutuhan...
4,MHS-005,Laki-laki,5,Teknik Informatika,44.4,37.5,93.8,3.1,Kurangnya arahan eksplorasi,Langsung diskusi dengan pihak industri bahwa s...


In [13]:
# 4. Simpan dataset bersih ke data/processed/student_features_clean.csv
out_dir = Path('../data/processed') if Path('../data').exists() else Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'student_features_clean.csv'
df_clean.to_csv(out_csv, index=False)
print(f'Selesai! Data tersimpan di: {out_csv}')

Selesai! Data tersimpan di: ..\data\processed\student_features_clean.csv
